In [73]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline # Still useful for organizing preprocessing

In [74]:
# --- Configuration ---
DATA_FILE_PATH = r'D:\GitHubRepos\is6400-business-data-analytics\data\loan_data.csv' # Use raw string for Windows paths
TARGET_COLUMN = 'loan_status'
TEST_SIZE = 0.2 # Proportion of data to use for testing
RANDOM_STATE = 42 # For reproducible splits
N_NEIGHBORS = 7 # Number of neighbors for KNN (can be tuned)

In [75]:
# --- 1. Load Data ---
print("Loading data...")
df = pd.read_csv(DATA_FILE_PATH)
print("Data loaded successfully.")

Loading data...
Data loaded successfully.


In [76]:
# --- 2. Initial Data Cleaning & Preparation ---
print("Performing initial data cleaning...")
# Handle potential outliers/errors identified in describe() - replace unrealistic values
# Cap age and experience.
df['person_age'] = df['person_age'].apply(lambda x: min(x, 100)) # Cap age at 100
df['person_emp_exp'] = df['person_emp_exp'].apply(lambda x: min(x, 60)) # Cap experience at 60

# Convert binary categorical feature explicitly
df['previous_loan_defaults_on_file'] = df['previous_loan_defaults_on_file'].map({'Yes': 1, 'No': 0}).astype(np.int8)

# Separate features (X) and target (y)
X = df.drop(TARGET_COLUMN, axis=1)
y = df[TARGET_COLUMN] # Target variable

# Identify column types for preprocessing
categorical_features = X.select_dtypes(include='object').columns.tolist()
numerical_features = X.select_dtypes(include=np.number).columns.tolist()

# Ensure the manually converted binary feature is treated as numerical
if 'previous_loan_defaults_on_file' in numerical_features:
    pass # Already numerical
elif 'previous_loan_defaults_on_file' in categorical_features:
    categorical_features.remove('previous_loan_defaults_on_file') # Remove if accidentally included

print(f"Categorical features: {categorical_features}")
print(f"Numerical features: {numerical_features}")

Performing initial data cleaning...
Categorical features: ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent']
Numerical features: ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'previous_loan_defaults_on_file']


In [77]:
# --- 3. Preprocessing Pipeline Definition ---
# Using Scikit-learn components
print("Defining preprocessing steps...")
preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    (StandardScaler(), numerical_features),
    remainder='passthrough' # Keep any columns not specified
)

Defining preprocessing steps...


In [78]:
# --- 4. Split Data ---
print("Splitting data into training and test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y # Good practice for classification
)

Splitting data into training and test sets...


In [79]:
# --- 5. Apply Preprocessing ---
# Fit the preprocessor on the training data ONLY and transform both sets
print("Applying preprocessing...")
# Fit on training data
preprocessor.fit(X_train)

# Transform training and test data
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Get feature names after transformation (optional, for context)
try:
    feature_names_out = preprocessor.get_feature_names_out()
    print(f"Number of features after preprocessing: {len(feature_names_out)}")
    # Convert processed arrays back to DataFrames with names (optional)
    X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names_out, index=X_train.index)
    X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names_out, index=X_test.index)
except AttributeError:
    print("Could not get feature names automatically (requires scikit-learn >= 1.0). Processed data are NumPy arrays.")
    # Output of transform will be NumPy arrays if get_feature_names_out fails or isn't used

Applying preprocessing...
Number of features after preprocessing: 26


In [80]:
# --- 6. Train KNN Model (CPU) ---
print(f"Training KNN model with k={N_NEIGHBORS} on CPU...")
knn_cpu = KNeighborsClassifier(n_neighbors=N_NEIGHBORS)
knn_cpu.fit(X_train_processed, y_train)
print("KNN model training complete.")

Training KNN model with k=7 on CPU...
KNN model training complete.


In [81]:
# --- 7. Make Predictions (CPU) ---
print("Making predictions on the test set (CPU)...")
y_pred = knn_cpu.predict(X_test_processed)
print("Predictions complete.")

Making predictions on the test set (CPU)...
Predictions complete.


In [82]:
# --- 8. Evaluate Model ---
print("\n--- Evaluation Metrics ---")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# For better visualization of the confusion matrix:
print("\nConfusion Matrix (labels):")
print("          Predicted 0  Predicted 1")
print(f"Actual 0:    {cm[0, 0]:<10} {cm[0, 1]:<10}")
print(f"Actual 1:    {cm[1, 0]:<10} {cm[1, 1]:<10}")

print("\n--- Script Finished ---")


--- Evaluation Metrics ---

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.94      7000
           1       0.80      0.72      0.76      2000

    accuracy                           0.90      9000
   macro avg       0.86      0.84      0.85      9000
weighted avg       0.90      0.90      0.90      9000


Confusion Matrix:
[[6648  352]
 [ 550 1450]]

Confusion Matrix (labels):
          Predicted 0  Predicted 1
Actual 0:    6648       352       
Actual 1:    550        1450      

--- Script Finished ---
